In [27]:
%cd /content/drive/MyDrive/Proyecto_Audio

/content/drive/MyDrive/Proyecto_Audio


In [28]:
!git status

On branch main
Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	deleted:    05_AudioSR.ipynb
	modified:   notebooks/06_ClearVoice_MossFormer2.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/05_AudioSR.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


In [29]:
!git add .
!git commit -m "Cambiar lo que he cambiado del drive de notebooks y colab notebooks"
!git push https://{token}@github.com/dehonidas9/TFG_restauracion_audio.git main

[main 0861679] Cambiar lo que he cambiado del drive de notebooks y colab notebooks
 2 files changed, 1 insertion(+), 1 deletion(-)
 rename 05_AudioSR.ipynb => notebooks/05_AudioSR.ipynb (100%)
 rewrite notebooks/06_ClearVoice_MossFormer2.ipynb (98%)
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 764 bytes | 382.00 KiB/s, done.
Total 4 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/dehonidas9/TFG_restauracion_audio.git
   68c07bb..0861679  main -> main


# 06_ClearVoice_MossFormer2.ipynb
### Modelo combinado multi-tarea (denoising + dereverb + super-resolución) — TFG Restauración interactiva de señales de audio degradadas

Modelo: **ClearVoice / MossFormer2** (ClearerVoice-Studio, Alibaba Tongyi Lab / DAMO Academy)

Repo: `modelscope/ClearerVoice-Studio`

A diferencia de los notebooks anteriores (un modelo especializado por categoría), este notebook usa **dos sub-modelos encadenados** de la misma familia MossFormer2, siguiendo el mismo patrón que documenta el propio equipo de ClearerVoice-Studio en su paper:

1. **MossFormer2_SE_48K** (speech enhancement) → reduce ruido y reverberación
2. **MossFormer2_SR_48K** (speech super-resolution) → reconstruye componentes de alta frecuencia

Esto es lo que en tu metodología se define como el "modelo combinado transversal a denoising, dereverberation y super-resolución", frente a los modelos especializados individuales (DeepFilterNet, MP-SENet, AudioSR) ya evaluados en notebooks anteriores.

Patrón del notebook:
1. Instalación de dependencias específicas
2. Imports y utils compartidos
3. Carga de los dos sub-modelos (SE y SR)
4. Selección de audio de test e inferencia encadenada (SE -> SR)
5. Métricas (bloque empírico: no-intrusivas)
6. Nota sobre la comparativa individual vs. combinado
7. Reproducción de audio (antes/después)
8. Liberar memoria GPU


## 1. Instalación de dependencias

In [ ]:
# ============================================================
# 06_ClearVoice_MossFormer2.ipynb — Instalacion
# ============================================================
# ClearVoice (ClearerVoice-Studio) empaqueta varios modelos de la
# familia MossFormer2 para SE, SR, SS y TSE bajo una API unificada.

!pip install clearvoice -q

# Nota: igual que en notebooks anteriores, si al instalar aparece
# conflicto de numpy/torch/torchaudio o pide reiniciar el runtime,
# reiniciar el kernel ANTES de ejecutar la celda de imports.

**⚠️ Aviso:** si el `pip install` falla o genera warnings de dependencias (numpy, torch, torchaudio), sigue el mismo protocolo de depuración que en los notebooks anteriores: mira el log completo (sin `-q` si hace falta), identifica el paquete conflictivo concreto, y decide si instalar con `--no-deps` + dependencias sueltas, en vez de asumir que funcionará a la primera.

## 2. Imports y utils compartidos

In [ ]:
import sys, os, time
sys.path.append('/content/drive/MyDrive/Proyecto_Audio/utils')

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
!pip install speechmos clearvoice onnxruntime pesq pystoi
import torch
import librosa
import soundfile as sf
import numpy as np
from clearvoice import ClearVoice

from audio_utils_funcionescomunes import cargar_audio, guardar_audio
from audio_utils_metricas_no_intrusivas import calcular_dnsmos
from audio_utils_metricas_ref import calcular_pesq, calcular_stoi, calcular_lsd
from audio_utils_memoria_GPU import liberar_memoria_gpu

BASE_DIR = '/content/drive/MyDrive/Proyecto_Audio'
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")

# Recordatorio de la sesion de AudioSR: comprobar SIEMPRE la firma real
# de las funciones de utils antes de asumir como llamarlas.
# import inspect; print(inspect.signature(calcular_dnsmos))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dispositivo: cuda


In [ ]:
#INICIO DE SESION TOKEN SECRETA DE COLAB (HUGGIN FACE PARA SOLICITUDES AUTENTICADAS A LA PAGINA)
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('TOKEN_TFG_CLEARMOSS'))



## 3. Carga de los sub-modelos (SE + SR)

In [ ]:
# Sub-modelo 1: Speech Enhancement (denoising + reduccion de reverberacion)
cv_se = ClearVoice(
    task='speech_enhancement',
    model_names=['MossFormer2_SE_48K']
)
print("MossFormer2_SE_48K cargado.")

# Sub-modelo 2: Speech Super-Resolution (reconstruccion de altas frecuencias)
cv_sr = ClearVoice(
    task='speech_super_resolution',
    model_names=['MossFormer2_SR_48K']
)
print("MossFormer2_SR_48K cargado.")

MossFormer2_SE_48K cargado.
downloading checkpoint for MossFormer2_SR_48K


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

MossFormer2_SR_48K cargado.


## 4. Selección de audio de test e inferencia encadenada (SE -> SR)

In [ ]:
# --- Seleccion del audio de entrada ---
carpeta_audio = '/content/drive/MyDrive/Proyecto_Audio/audio_samples'
print(f"Archivos disponibles en {carpeta_audio}:\n")
for f in os.listdir(carpeta_audio):
    print(f" - {f}")

nombre_archivo = input("\nIntroduce el nombre exacto del archivo a usar (con extension): ")
input_path = os.path.join(carpeta_audio, nombre_archivo)

if not os.path.isfile(input_path):
    raise FileNotFoundError(f"No se encontro el archivo: {input_path}")

print(f"\nUsando como input: {input_path}")

output_dir = f'{BASE_DIR}/outputs'
os.makedirs(output_dir, exist_ok=True)

path_intermedio_se = f'{output_dir}/clearvoice_intermedio_SE.wav'
output_path = f'{output_dir}/clearvoice_final_SE_SR.wav'

t0 = time.time()

# --- Paso 1: Speech Enhancement (denoising + dereverb) ---
print("Ejecutando MossFormer2_SE_48K (enhancement)...")
output_wav_se = cv_se(input_path=input_path, online_write=False)
cv_se.write(output_wav_se, output_path=path_intermedio_se)
print(f"  -> Guardado intermedio en: {path_intermedio_se}")

# --- Paso 2: Speech Super-Resolution sobre la salida ya mejorada ---
print("Ejecutando MossFormer2_SR_48K (super-resolucion)...")
output_wav_sr = cv_sr(input_path=path_intermedio_se, online_write=False)
cv_sr.write(output_wav_sr, output_path=output_path)
print(f"  -> Guardado final en: {output_path}")

print(f"\nPipeline combinado completado en {time.time()-t0:.1f}s")

Archivos disponibles en /content/drive/MyDrive/Proyecto_Audio/audio_samples:

 - AUDIO_REVERB_ALBIOL_TFG.wav

Introduce el nombre exacto del archivo a usar (con extension): AUDIO_REVERB_ALBIOL_TFG.wav

Usando como input: /content/drive/MyDrive/Proyecto_Audio/audio_samples/AUDIO_REVERB_ALBIOL_TFG.wav
Ejecutando MossFormer2_SE_48K (enhancement)...
Running MossFormer2_SE_48K ...



100%|██████████| 1/1 [00:18<00:00, 18.02s/it]


  -> Guardado intermedio en: /content/drive/MyDrive/Proyecto_Audio/outputs/clearvoice_intermedio_SE.wav
Ejecutando MossFormer2_SR_48K (super-resolucion)...
Running MossFormer2_SR_48K ...



100%|██████████| 1/1 [00:49<00:00, 49.13s/it]

  -> Guardado final en: /content/drive/MyDrive/Proyecto_Audio/outputs/clearvoice_final_SE_SR.wav

Pipeline combinado completado en 67.6s


**Nota técnica:** este es el mismo orden de encadenado (SE -> SR) que documenta el propio equipo de ClearerVoice-Studio en su paper, justificado porque el ruido de fondo degrada la estimación espectral que hace el modelo de SR. Si tu audio de test no tiene ruido significativo, puedes probar también el orden inverso o solo SR, y comparar — puede ser un dato interesante para la sección de resultados (ablation del orden del pipeline).

Ambos submodelos trabajan a 48kHz internamente; si tu audio de entrada tiene un sample rate distinto, ClearVoice lo resamplea automáticamente (a diferencia de AudioSR, no hace falta que lo hagas tú a mano).

## 5. Métricas (bloque empírico: no-intrusivas)

In [ ]:
# DNSMOS sobre el resultado final (SE+SR encadenados)
audio_out, sr_out = librosa.load(output_path, sr=None, mono=True)
dnsmos_final = calcular_dnsmos(audio_out, sr_out)
print(f"DNSMOS (ClearVoice SE+SR combinado): {dnsmos_final}")

# Tambien es interesante medir el resultado intermedio (solo SE, antes de SR)
# para poder aislar la aportacion de cada submodelo en la comparativa
audio_intermedio, sr_intermedio = librosa.load(path_intermedio_se, sr=None, mono=True)
dnsmos_intermedio = calcular_dnsmos(audio_intermedio, sr_intermedio)
print(f"DNSMOS (solo SE, antes de SR): {dnsmos_intermedio}")

DNSMOS (ClearVoice SE+SR combinado): {'ovrl_mos': 2.08605964602275, 'sig_mos': 2.2816471375628082, 'bak_mos': 3.522901489816461, 'p808_mos': 2.2335253}
DNSMOS (solo SE, antes de SR): {'ovrl_mos': 2.094748758127067, 'sig_mos': 2.3066022140428184, 'bak_mos': 3.560333689265572, 'p808_mos': 2.339101}


## 6. Nota: comparativa individual vs. combinado

In [ ]:
# Este notebook NO corre un baseline clasico propio: al ser el modelo
# combinado (transversal a 3 categorias), su punto de comparacion no es
# un baseline no-IA, sino los modelos INDIVIDUALES ya evaluados:
#   - Denoising: DeepFilterNet2/3 (notebook 01)
#   - Dereverberation: MP-SENet (notebook 04)
#   - Super-resolucion: AudioSR (notebook 05)
#
# La comparativa "individual vs. combinado" (una de las comparativas
# cruzadas de la metodologia) se hace en la fase de analisis/redaccion,
# cruzando los DNSMOS (y metricas bibliograficas) de cada notebook individual
# frente al DNSMOS obtenido aqui con ClearVoice.

print("Resumen de DNSMOS obtenidos en este notebook:")
print(f"  - Solo SE (denoising+dereverb parcial): {dnsmos_intermedio}")
print(f"  - SE + SR combinado (pipeline completo): {dnsmos_final}")

Resumen de DNSMOS obtenidos en este notebook:
  - Solo SE (denoising+dereverb parcial): {'ovrl_mos': 2.094748758127067, 'sig_mos': 2.3066022140428184, 'bak_mos': 3.560333689265572, 'p808_mos': 2.339101}
  - SE + SR combinado (pipeline completo): {'ovrl_mos': 2.08605964602275, 'sig_mos': 2.2816471375628082, 'bak_mos': 3.522901489816461, 'p808_mos': 2.2335253}


## 7. Reproducción de audio (verificación manual)

In [ ]:
from IPython.display import Audio, display

print("Audio original (degradado):")
display(Audio(input_path))

print("Audio tras SE (denoising + dereverb):")
display(Audio(path_intermedio_se))

print("Audio tras SE + SR (pipeline combinado completo):")
display(Audio(output_path))

Output hidden; open in https://colab.research.google.com to view.

## 8. Liberar memoria GPU

In [ ]:
liberar_memoria_gpu(cv_se)
liberar_memoria_gpu(cv_sr)
torch.cuda.empty_cache()
print("Memoria GPU liberada.")

Aviso: sin nombre_variable, solo se libera la referencia local a esta función. Si el modelo sigue asignado a una variable en el notebook (ej. modelo_dfn), la GPU no se liberará del todo. Llama a esta función como liberar_memoria_gpu(modelo_dfn, 'modelo_dfn') para liberarla de verdad.
Memoria GPU liberada. Uso actual: 0.68 GB
Aviso: sin nombre_variable, solo se libera la referencia local a esta función. Si el modelo sigue asignado a una variable en el notebook (ej. modelo_dfn), la GPU no se liberará del todo. Llama a esta función como liberar_memoria_gpu(modelo_dfn, 'modelo_dfn') para liberarla de verdad.
Memoria GPU liberada. Uso actual: 0.68 GB
Memoria GPU liberada.


---
### Notas / posibles problemas a vigilar

1. **Conflictos de dependencias al instalar** `clearvoice` (numpy/torch/torchaudio) — mismo protocolo que en notebooks anteriores: mirar el log completo, identificar el paquete concreto, decidir si hace falta `--no-deps`.
2. **Nombres/firmas de funciones de utils**: recuerda comprobar con `inspect.signature()` antes de asumir cómo llamarlas (ya nos pasó con `calcular_dnsmos` y `baseline_bwe_interpolacion_spline` en el notebook de AudioSR).
3. **Encadenado SE->SR**: si el resultado final suena peor que el intermedio (solo SE), puede deberse a que el SR está sobre-procesando una señal ya limpia — vale la pena probar el pipeline en ambos órdenes con tu audio real y quedarte con el que dé mejor DNSMOS, documentando la decisión en la memoria.
4. Este notebook cierra la Fase 2 (todos los modelos ya evaluados). Los siguientes pasos son la Fase 3 (recopilar métricas bibliográficas publicadas de cada modelo) y la Fase 4 (integración Gradio de dos niveles).
